In [22]:
import re
import logging
from pathlib import Path
import pandas as pd
import pdfplumber

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

In [23]:
PDF_PATH = "../data/BOFA_072025_0504.pdf"
# PDF_PATH = "../data/Marcus_debit.PDF"

`extract_text` — pull raw text from every PDF page

In [24]:
def extract_text(pdf_path: str) -> str:
    """Extract all text from a PDF using pdfplumber."""
    pages_text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ""
            pages_text.append(text)
    return "\n".join(pages_text)

# --- test ---
text = extract_text(PDF_PATH)
print(text[:2000])

Customer Service Information:
www.bankofamerica.com
1.800.421.2110
Mail billing inquiries to:
P.O.BOX15284 Bank of America
WILMINGTON,DE 19850 P.O. Box 672050
Dallas TX 75267-2050
Mail payment to:
Bank of America
P.O. Box 15019
Wilmington DE 19886-5019
WILSON JIANG
135 CHRYSTIE ST APT A
NEW YORK NY 10002-2831
Visa Signature®
Account# 4400 6673 0259 0504
August 7 - September 6, 2025
Account Summary/Payment Information New Balance Total $1,637.19
Current Payment Due $35.00
Previous Balance $0.00
Total Minimum Payment Due $35.00
Payments and Other Credits $0.00
Payment Due Date 10/03/2025
Purchases and Adjustments $1,637.19
Fees Charged $0.00
Interest Charged $0.00
Late Payment Warning: If we do not receive your Total Minimum
New Balance Total $1,637.19
Payment by the date listed above, you may have to pay a late fee of up to
Total Credit Line $4,000.00 $40.00 and your APRs may be increased up to the Penalty APR of 29.99%.
Total Credit Available $2,362.81 Total Minimum Payment Warning: If

`classify_account_type` — weighted regex scoring to detect credit / checking / savings

In [25]:
_ACCOUNT_SIGNALS: dict[str, list[tuple[re.Pattern, int]]] = {
    "credit": [
        (re.compile(r"minimum\s+payment\s+due",                  re.I), 3),
        (re.compile(r"credit\s+limit",                           re.I), 3),
        (re.compile(r"available\s+credit",                       re.I), 3),
        (re.compile(r"cash\s+advance",                           re.I), 3),
        (re.compile(r"purchase\s+apr",                           re.I), 3),
        (re.compile(r"payments.{0,10}credits.{0,10}adjustments", re.I), 2),
        (re.compile(r"rewards?\s+points?",                       re.I), 2),
        (re.compile(r"statement\s+balance",                      re.I), 1),
    ],
    "checking": [
        (re.compile(r"checks?\s+paid",                           re.I), 3),
        (re.compile(r"checking\s+account",                       re.I), 3),
        (re.compile(r"debit\s+card\s+purchases?",                re.I), 2),
        (re.compile(r"\boverdraft\b",                            re.I), 2),
        (re.compile(r"deposits?\s+and\s+additions",              re.I), 2),
        (re.compile(r"atm\s+withdrawal",                         re.I), 1),
        (re.compile(r"direct\s+deposit",                         re.I), 1),
    ],
    "savings": [
        (re.compile(r"online\s+savings",                         re.I), 3),
        (re.compile(r"high.yield\s+savings",                     re.I), 3),
        (re.compile(r"savings\s+account",                        re.I), 3),
        (re.compile(r"annual\s+percentage\s+yield",              re.I), 2),
        (re.compile(r"money\s+market",                           re.I), 2),
        (re.compile(r"interest\s+earned",                        re.I), 1),
        (re.compile(r"interest\s+(paid|credited)",               re.I), 1),
    ],
}


def classify_account_type(text: str) -> str | None:
    """Classify account type by scoring weighted regex signal hits across the full statement text."""
    scores = {account_type: 0 for account_type in _ACCOUNT_SIGNALS}
    for account_type, signals in _ACCOUNT_SIGNALS.items():
        for pattern, weight in signals:
            if pattern.search(text):
                scores[account_type] += weight

    best_type, best_score = max(scores.items(), key=lambda x: x[1])
    return best_type if best_score > 0 else None

# --- test ---
account_type = classify_account_type(text)
print("account_type:", account_type)

account_type: credit


`extract_last_four` — pull last 4 digits of account number from the statement header

In [26]:
_LAST_FOUR_RE = re.compile(
    r'(?:'
    r'ending\s+in\s+(\d{4})'
    r'|account\s*(?:number|no\.?|#)[:\s]+[ \d*xX-]*?(\d{4})\b'
    r'|\*{2,}(\d{4})\b'
    r'|[xX]{2,}(\d{4})\b'
    r'|(?:checking|savings)\s+\d*(\d{4})\b'
    r')',
    re.I,
)


def extract_last_four(text: str) -> str | None:
    """Return the last 4 digits of the account number from the statement header, or None."""
    account_match = _LAST_FOUR_RE.search(text[:3000])
    # print(account_match.groups())
    if account_match:
        return next(capture for capture in account_match.groups() if capture is not None)
    return None

# --- test ---
last_four = extract_last_four(text)
print("last_four:", last_four)

last_four: 4400


`extract_card_name` — identify the card/account product name from the header

In [27]:
_MARCUS_ACCOUNT_RE = re.compile(r'AccountName\s+([A-Za-z]+)', re.I)

_CARD_NAME_SIGNALS: list[tuple[re.Pattern, str]] = [
    # Chase credit
    (re.compile(r'sapphire\s+reserve',          re.I), "Chase Sapphire Reserve"),
    (re.compile(r'sapphire\s+preferred',        re.I), "Chase Sapphire Preferred"),
    (re.compile(r'sapphire',                    re.I), "Chase Sapphire"),
    (re.compile(r'freedom\s+unlimited',         re.I), "Chase Freedom Unlimited"),
    (re.compile(r'freedom\s+flex',              re.I), "Chase Freedom Flex"),
    (re.compile(r'freedom\s+rise',              re.I), "Chase Freedom Rise"),
    (re.compile(r'freedom',                     re.I), "Chase Freedom"),
    # Chase checking
    (re.compile(r'college\s+checking',          re.I), "Chase College Checking"),
    (re.compile(r'sapphire\s+checking',         re.I), "Chase Sapphire Checking"),
    (re.compile(r'premier\s+plus\s+checking',   re.I), "Chase Premier Plus Checking"),
    (re.compile(r'total\s+checking',            re.I), "Chase Total Checking"),
    # Capital One
    (re.compile(r'venture\s*one',               re.I), "Capital One VentureOne"),
    (re.compile(r'venture\s*x',                 re.I), "Capital One Venture X"),
    (re.compile(r'venture',                     re.I), "Capital One Venture"),
    (re.compile(r'quicksilver\s*one',           re.I), "Capital One QuicksilverOne"),
    (re.compile(r'quicksilver',                 re.I), "Capital One Quicksilver"),
    (re.compile(r'savor\s*one',                 re.I), "Capital One SavorOne"),
    (re.compile(r'savor',                       re.I), "Capital One Savor"),
    (re.compile(r'spark',                       re.I), "Capital One Spark"),
    # Bank of America
    (re.compile(r'customized\s+cash\s+rewards', re.I), "Bank of America Customized Cash Rewards"),
    (re.compile(r'unlimited\s+cash\s+rewards',  re.I), "Bank of America Unlimited Cash Rewards"),
    (re.compile(r'premium\s+rewards',           re.I), "Bank of America Premium Rewards"),
    (re.compile(r'travel\s+rewards',            re.I), "Bank of America Travel Rewards"),
    (re.compile(r'cash\s+rewards',              re.I), "Bank of America Cash Rewards"),
    (re.compile(r'visa\s+signature',            re.I), "Bank of America Visa Signature"),
    # Wells Fargo
    (re.compile(r'active\s+cash',               re.I), "Wells Fargo Active Cash"),
    (re.compile(r'autograph',                   re.I), "Wells Fargo Autograph"),
    (re.compile(r'reflect',                     re.I), "Wells Fargo Reflect"),
    # American Express
    (re.compile(r'platinum\s+card',             re.I), "Amex Platinum"),
    (re.compile(r'gold\s+card',                 re.I), "Amex Gold"),
    (re.compile(r'blue\s+cash\s+preferred',     re.I), "Amex Blue Cash Preferred"),
    (re.compile(r'blue\s+cash\s+everyday',      re.I), "Amex Blue Cash Everyday"),
    # Citi
    (re.compile(r'double\s+cash',               re.I), "Citi Double Cash"),
    (re.compile(r'custom\s+cash',               re.I), "Citi Custom Cash"),
    (re.compile(r'strata\s+premier',            re.I), "Citi Strata Premier"),
    # Discover
    (re.compile(r'discover\s+it',               re.I), "Discover It"),
]


def extract_card_name(text: str) -> str:
    """Return the card or account product name from the statement header."""
    header = text[:3000]

    marcus_match = _MARCUS_ACCOUNT_RE.search(header)
    if marcus_match:
        raw = marcus_match.group(1)
        name = re.sub(r'(?<=[a-z])(?=[A-Z])', ' ', raw)
        return f"Marcus {name}"

    best_pos, best_name = len(header) + 1, None
    for pattern, name in _CARD_NAME_SIGNALS:
        match = pattern.search(header)
        if match and match.start() < best_pos:
            best_pos, best_name = match.start(), name

    if best_name is None:
        return "Unknown Account"

    return best_name

# --- test ---
card_name = extract_card_name(text)
print("card_name:", card_name)

card_name: Bank of America Visa Signature


`extract_statement_period` — parse the billing period dates from the header

In [28]:
_DATE_PAT = (
    r'\d{1,2}/\d{1,2}/\d{2,4}'
    r'|(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?'
    r'|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)'
    r'\s+\d{1,2},?\s+\d{4}'
)

_DATE_NO_YEAR_PAT = (
    r'(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?'
    r'|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)'
    r'\s+\d{1,2},?'
)

_PERIOD_RE = re.compile(rf'({_DATE_PAT})\s*(?:through|to|[-–])\s*({_DATE_PAT})', re.I)
_PERIOD_SHARED_YEAR_RE = re.compile(rf'({_DATE_NO_YEAR_PAT})\s*[-–]\s*({_DATE_PAT})', re.I)


def extract_statement_period(text: str) -> dict:
    """Return {'period_start': 'YYYY-MM-DD', 'period_end': 'YYYY-MM-DD'} or Nones."""
    period_match = _PERIOD_RE.search(text[:3000])
    if period_match:
        try:
            return {
                "period_start": pd.to_datetime(period_match.group(1)).strftime("%Y-%m-%d"),
                "period_end":   pd.to_datetime(period_match.group(2)).strftime("%Y-%m-%d"),
            }
        except Exception:
            pass

    # Fallback: "Month Day - Month Day, Year" where year only appears on the end date (BofA)
    shared_year_match = _PERIOD_SHARED_YEAR_RE.search(text[:3000])
    if shared_year_match:
        try:
            end_date = pd.to_datetime(shared_year_match.group(2))
            start_date = pd.to_datetime(f"{shared_year_match.group(1)} {end_date.year}")
            if start_date > end_date:
                start_date = start_date.replace(year=end_date.year - 1)
            return {
                "period_start": start_date.strftime("%Y-%m-%d"),
                "period_end":   end_date.strftime("%Y-%m-%d"),
            }
        except Exception:
            pass

    return {"period_start": None, "period_end": None}

# --- test ---
period = extract_statement_period(text)
print("period:", period)

period: {'period_start': '2025-08-07', 'period_end': '2025-09-06'}


`extract_transactions` — parse transaction rows into a DataFrame

In [29]:
_DATE_PATTERN = (
    r'\b\d{1,2}/\d{1,2}(?:/\d{2,4})?\b|'
    r'\b(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|'
    r'May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|'
    r'Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)\s+\d{1,2}\b'
)
_AMOUNT_PATTERN = r'-?\$?\d{1,3}(?:,\d{3})*(?:\.\d{2})?'

_TRANSACTION_ROW = re.compile(
    rf'^\s*({_DATE_PATTERN})'
    rf'(?:\s+({_DATE_PATTERN}))?'
    rf'\s+(.*?)'
    rf'\s+({_AMOUNT_PATTERN})'
    rf'(?:\s+({_AMOUNT_PATTERN}))?'
    rf'\s*$',
    re.I,
)

_NEGATIVE_SECTION_KEYWORDS = ["PAYMENTS, CREDITS AND ADJUSTMENTS"]

_CC_PAYMENT_PATTERN = re.compile(
    r'\b(autopay|pymt|payment|ach\s+transfer)\b',
    re.I,
)


def extract_transactions(bank_text: str) -> pd.DataFrame:
    """Parse transaction rows from extracted PDF text into a DataFrame."""
    transactions = []
    is_negative_section = False

    for line in bank_text.splitlines():
        line = " ".join(line.split())
        if not line:
            continue

        if any(keyword in line.upper() for keyword in _NEGATIVE_SECTION_KEYWORDS):
            is_negative_section = True
            continue
        elif "TRANSACTIONS" in line.upper():
            is_negative_section = False
            continue

        match = _TRANSACTION_ROW.match(line)
        if match:
            trans_date, post_date, description, amount1, amount2 = match.groups()
            description = description.strip()

            if is_negative_section and _CC_PAYMENT_PATTERN.search(description):
                continue

            amount1_num = float(amount1.replace("$", "").replace(",", ""))
            amount2_num = float(amount2.replace("$", "").replace(",", "")) if amount2 else None

            if is_negative_section:
                amount1_num = -abs(amount1_num)
                if amount2_num is not None:
                    amount2_num = -abs(amount2_num)

            transactions.append({
                "trans_date":  trans_date,
                "description": description,
                "amount1":     amount1_num,
                "amount2":     amount2_num,
            })

    return pd.DataFrame(transactions)

# --- test ---
df = extract_transactions(text)
print(f"{len(df)} transactions found")
df

6 transactions found


,trans_date,description,amount1,amount2
0,09/04,UNIVERSAL ORLANDO WEBSIT 407-224-4233 FL 9827 ...,1192.76,None
1,09/05,AIRBNB * HM3DXD8W3T AIRBNB.COM CA 2453 0504,444.43,None
2,09/06,INTEREST CHARGED ON PURCHASES,0.00,None
3,09/06,INTEREST CHARGED ON BALANCE TRANSFERS,0.00,None
4,09/06,INTEREST CHARGED ON DIR DEP&CHK CASHADV,0.00,None
5,09/06,INTEREST CHARGED ON BANK CASH ADVANCES,0.00,None


`parse_pdf` — orchestrates all of the above into a single result dict

In [30]:
def parse_pdf(pdf_path: str) -> dict:
    text = extract_text(pdf_path)
    df = extract_transactions(text)
    period = extract_statement_period(text)
    account_type = classify_account_type(text)
    last_four = extract_last_four(text)
    card_name = extract_card_name(text)

    logger.info(
        "Parsed %s: %d transactions, period %s → %s, account_type %s, last_four %s, card_name %s",
        Path(pdf_path).name, len(df),
        period["period_start"], period["period_end"],
        account_type, last_four, card_name,
    )

    return {
        "period_start": period["period_start"],
        "period_end":   period["period_end"],
        "account_type": account_type,
        "last_four":    last_four,
        "card_name":    card_name,
        "transactions": df,
    }

# --- test ---
result = parse_pdf(PDF_PATH)
print({k: v for k, v in result.items() if k != "transactions"})
result["transactions"]

INFO | Parsed BOFA_072025_0504.pdf: 6 transactions, period 2025-08-07 → 2025-09-06, account_type credit, last_four 4400, card_name Bank of America Visa Signature


{'period_start': '2025-08-07', 'period_end': '2025-09-06', 'account_type': 'credit', 'last_four': '4400', 'card_name': 'Bank of America Visa Signature'}


,trans_date,description,amount1,amount2
0,09/04,UNIVERSAL ORLANDO WEBSIT 407-224-4233 FL 9827 ...,1192.76,None
1,09/05,AIRBNB * HM3DXD8W3T AIRBNB.COM CA 2453 0504,444.43,None
2,09/06,INTEREST CHARGED ON PURCHASES,0.00,None
3,09/06,INTEREST CHARGED ON BALANCE TRANSFERS,0.00,None
4,09/06,INTEREST CHARGED ON DIR DEP&CHK CASHADV,0.00,None
5,09/06,INTEREST CHARGED ON BANK CASH ADVANCES,0.00,None


In [32]:
parse_pdf(PDF_PATH)

INFO | Parsed BOFA_072025_0504.pdf: 6 transactions, period 2025-08-07 → 2025-09-06, account_type credit, last_four 4400, card_name Bank of America Visa Signature


{'period_start': '2025-08-07',
 'period_end': '2025-09-06',
 'account_type': 'credit',
 'last_four': '4400',
 'card_name': 'Bank of America Visa Signature',
 'transactions':   trans_date                                        description  amount1  \
 0      09/04  UNIVERSAL ORLANDO WEBSIT 407-224-4233 FL 9827 ...  1192.76   
 1      09/05        AIRBNB * HM3DXD8W3T AIRBNB.COM CA 2453 0504   444.43   
 2      09/06                      INTEREST CHARGED ON PURCHASES     0.00   
 3      09/06              INTEREST CHARGED ON BALANCE TRANSFERS     0.00   
 4      09/06            INTEREST CHARGED ON DIR DEP&CHK CASHADV     0.00   
 5      09/06             INTEREST CHARGED ON BANK CASH ADVANCES     0.00   
 
   amount2  
 0    None  
 1    None  
 2    None  
 3    None  
 4    None  
 5    None  }